# Calc Modes Demo

Demonstrates the completed `arb_calc` and `acb_calc` functions with the four modes:

- `point`
- `basic`
- `adaptive`
- `rigorous`


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists() and (p / 'tools').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('repo:', REPO_ROOT)


In [ ]:
# Choose backend mode before importing JAX modules
import os
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
if JAX_MODE not in {'cpu', 'gpu'}:
    raise ValueError(f'JAX_MODE must be cpu or gpu, got: {JAX_MODE}')
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
if JAX_DTYPE not in {'float64', 'float32'}:
    raise ValueError(f'JAX_DTYPE must be float64 or float32, got: {JAX_DTYPE}')
os.environ['JAX_PLATFORMS'] = 'cuda' if JAX_MODE == 'gpu' else 'cpu'
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)


In [2]:
import jax.numpy as jnp

from arbplusjax import acb_core, api, calc_wrappers, double_interval as di


ModuleNotFoundError: No module named 'arbplusjax'

In [ ]:
JAX_SCALAR_DTYPE = getattr(jnp, JAX_DTYPE)

def iv(lo, hi):
    return di.interval(JAX_SCALAR_DTYPE(lo), JAX_SCALAR_DTYPE(hi))

def box(re_lo, re_hi, im_lo, im_hi):
    return acb_core.acb_box(iv(re_lo, re_hi), iv(im_lo, im_hi))

a = iv(-0.2, 0.1)
b = iv(0.7, 0.9)
z0 = box(-0.2, 0.1, -0.05, 0.05)
z1 = box(0.7, 0.9, 0.1, 0.2)


## Point Mode via API

In [ ]:
api.eval_point("arb_calc_partition", a, b, 6)

## Interval Modes via Generated Calc Wrappers

In [ ]:
for mode in ("basic", "adaptive", "rigorous"):
    out = calc_wrappers.arb_calc_partition_mode(a, b, 6, impl=mode, prec_bits=170)
    print(mode, out.shape, out[:2])

In [ ]:
for mode in ("basic", "adaptive", "rigorous"):
    out = calc_wrappers.acb_calc_integrate_mode(z0, z1, impl=mode, prec_bits=170)
    print(mode, out)

## New Calc APIs

In [ ]:
print("arb_calc_newton_step", api.eval_interval("arb_calc_newton_step", iv(0.2, 0.4), mode="basic", prec_bits=170))
print("arb_calc_refine_root_newton", api.eval_interval("arb_calc_refine_root_newton", iv(0.2, 0.4), mode="basic", prec_bits=170))
print("arb_calc_isolate_roots", api.eval_interval("arb_calc_isolate_roots", a, b, 6, mode="basic", prec_bits=170).shape)
print("acb_calc_cauchy_bound", api.eval_interval("acb_calc_cauchy_bound", z0, z1, mode="basic", prec_bits=170))
print("acb_calc_integrate_gl_auto_deg", api.eval_interval("acb_calc_integrate_gl_auto_deg", z0, z1, mode="basic", prec_bits=170))
print("acb_calc_integrate_taylor", api.eval_interval("acb_calc_integrate_taylor", z0, z1, mode="basic", prec_bits=170))